# Function Calling / Tool Calling

В этом ноутбуке мы попробуем создать агента, который будет записывать сделанные нами упражнения. Для этого мы будем использовать **вызов инструментов** (Tool Calling или Function Calling) 

Современные языковые модели знают многое, но они не могут выполнять конкретные действия: знать текущую погоду или сохранять информацию в базу данных. Function Calling решает эту проблему — он позволяет модели делегировать такие задачи специальным инструментам.

> **💡 Function Calling** (или вызов функций) — это способность языковой модели определять, когда и какую функцию следует вызвать в ответ на запрос пользователя, а также формировать корректные аргументы для этой функции.

## Как работает Function Calling

Рассмотрите процесс работы Function Calling на простом примере с фитнес-ассистентом:

- **Вы описываете для модели доступные инструменты** — например, функцию записи упражнения с параметрами: название упражнения, количество подходов и повторений.

- **Пользователь задаёт вопрос** — например, **Запиши, что я сегодня сделал 3 подхода по 12 приседаний**.

- **Модель анализирует запрос и возвращает структуру для вызова функции**: в примере параметры для функции записи упражнения: `{"exercise_name": "приседания", "sets": 3, "reps": 12}`. Ответ модели будет содержать объект с этим вызовом.

- **Ваше приложение получает результат от модели, выполняет функцию** — записывает данные о тренировке в журнал и возвращает статус операции, например: `{"status": "success", "message": "Упражнение записано в дневник тренировок"}`.

- **Модель генерирует итоговый ответ пользователю** на основе результата выполнения функции: **Отлично! Я записал твои приседания: 3 подхода по 12 повторений. Хорошая работа! Хочешь добавить ещё какие-то упражнения в сегодняшнюю тренировку?**

Благодаря такому механизму ассистент не просто имитирует выполнение действий, а реально сохраняет информацию, которую можно использовать в дальнейшем, например для отслеживания прогресса тренировок.

Модель не взаимодействует с внешними функциями напрямую и не вызывает их сама. Она структурирует свои намерения в специальном формате, а дальше обработка лежит на стороне разработчика.

## Когда использовать Function Calling

Function Calling пригодится, когда нужно:

- Получить актуальную информацию: курсы валют, расписание, прогноз погоды.
- Выполнить точные вычисления, где нужен результат именно расчётов (а не приближение модели).
- Автоматизировать рабочие процессы через CRM/ERP-системы.
- Вести трекинг задач, назначать поручения.
 Управлять удалёнными устройствами, сервисами (например, запускать бэкапы).
- Генерировать и визуализировать структурированные данные (таблицы, отчёты, графики).

## Как реализовать Function Calling для фитнес-ассистента

Теперь, когда мы понимаем принцип работы Function Calling, давайте реализуем агента, который будет записывать сделанные упражнения. Это позволит нам создать простую, но полезную систему учёта тренировок. Сделав для начала отдельного агента, мы затем сможем добавить эту функциональность основному ассистенту.

Для начала импортируем библиотеки и создадим клиента OpenAI:


In [ ]:
%pip install openai dotenv

In [1]:
import openai
import os
import json
import uuid
from datetime import datetime, timedelta
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]


model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

### 1️⃣ Определение функции для записи упражнений

Фитнес-ассистент должен сохранять информацию о том, что пользователь уже сделал в ходе тренировок. Например, когда пользователь говорит: **Сегодня я сделал 3 подхода по 12 приседаний с весом 70 кг**, ассистент должен не только похвалить за выполненную работу, но и сохранить эти данные для дальнейшего анализа прогресса.

Сначала нужно определить функцию, которая будет записывать информацию о выполненных упражнениях. Нужно сохранять запись о том, какое упражнение было выполнено, с каким весом и количеством повторений:

In [2]:
import uuid

from datetime import datetime


# Простое хранилище данных в памяти
exercises_db = {}


def log_exercise(exercise_name, sets, reps, weight=None, date=None):
    """
    Записывает информацию о выполненном упражнении в журнал тренировок
    
    Args:
        exercise_name (str): Название упражнения
        sets (int): Количество подходов
        reps (int): Количество повторений в каждом подходе
        weight (float, optional): Вес в кг
        date (str, optional): Дата тренировки в формате YYYY-MM-DD
    
    Returns:
        dict: Информация о записи с уникальным ID
    """
    # Генерируем уникальный ID для записи
    record_id = str(uuid.uuid4())
    
    # Устанавливаем текущую дату, если не указана
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')
    
    # Создаём запись
    record = {
        "id": record_id,
        "exercise": exercise_name,
        "sets": sets,
        "reps": reps,
        "weight": weight,
        "date": date
    }
    
    # В реальном приложении здесь был бы код для сохранения в базу данных
    # Для примера просто сохраняем в памяти
    if 'exercise_log' not in exercises_db:
        exercises_db['exercise_log'] = []
    
    exercises_db['exercise_log'].append(record)
    
    return {
        "status": "success",
        "message": f"Упражнение '{exercise_name}' успешно записано",
        "record_id": record_id
    }



### 2️⃣ Описание функции для языковой модели в Responses API

После определения функции нужно создать её описание для языковой модели. Такая информация позволит модели понять, когда и как вызывать эту функцию. В Responses API это делается с помощью специальной структуры данных:

In [3]:
# Описание функции для Responses API

log_exercise_tool = {
    "type": "function",
    "name": "log_exercise",
    "description": "Записывает информацию о выполненном упражнении в журнал тренировок",
    "parameters": {
        "type": "object",
        "properties": {
            "exercise_name": {
                "type": "string",
                "description": "Название упражнения"
            },
            "sets": {
                "type": "integer",
                "description": "Количество подходов"
            },
            "reps": {
                "type": "integer",
                "description": "Количество повторений в каждом подходе"
            },
            "weight": {
                "type": "number",
                "description": "Вес в кг (если применимо)"
            },
            "date": {
                "type": "string",
                "description": "Дата тренировки в формате YYYY-MM-DD (если не указана, используется сегодняшняя дата)"
            }
        },
        "required": ["exercise_name", "sets", "reps"]
    }
}


Эта структура использует формат JSON Schema для определения параметров функции. Обратите внимание на ключевые элементы:

- `type: "function"` — указывает, что это описание функции;

- `name` — имя функции, которое будет использоваться при вызове;

- `description` — помогает модели понять назначение функции и когда её вызывать;

- `parameters` — схема параметров, которая включает:

    - `properties` — описание каждого параметра с его типом и назначением;

    - `required` — список обязательных параметров.

Качественное описание функции критически важно — оно напрямую влияет на то, насколько точно модель будет определять необходимость её вызова и корректно передавать параметры.

### 3️⃣ Вызов модели с инструментом

Теперь описание функции можно передать языковой модели при вызове. Это объяснит модели, что ей доступен дополнительный инструмент для записи упражнений:

In [8]:
# Вызываем модель с доступным инструментом
res = client.responses.create(
    model=model,
    instructions="Ты — профессиональный фитнес-ассистент. Помогаешь пользователю вести дневник тренировок.",
    input="Я сегодня сделал 3 подхода по 12 приседаний с весом 70 кг",
    tools=[log_exercise_tool]  # Передаём наш инструмент
)

res.dict()

C:\Users\dmitr\AppData\Local\Temp\ipykernel_9336\3957113112.py:9: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  res.dict()


{'id': '60cdc911-8ee4-4b5a-9abf-57589b1ea265',
 'created_at': 1772898462.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'Ты — профессиональный фитнес-ассистент. Помогаешь пользователю вести дневник тренировок.',
 'metadata': {},
 'model': 'gpt://b1gkp4q6n092isf1ee3o/qwen3-235b-a22b-fp8/latest',
 'object': 'response',
 'output': [{'arguments': '{"exercise_name": "приседания", "sets": 3, "reps": 12, "weight": 70}',
   'call_id': 'chatcmpl-tool-bb8df507af1d4c428a655a9a45aacb3a',
   'name': 'log_exercise',
   'type': 'function_call',
   'id': 'baed7753-036f-4da3-a300-63b0237207b6',
   'status': 'completed',
   'valid': True}],
 'parallel_tool_calls': False,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [{'name': 'log_exercise',
   'parameters': {'type': 'object',
    'properties': {'exercise_name': {'type': 'string',
      'description': 'Название упражнения'},
     'sets': {'type': 'integer', 'description': 'Количество подходов'},
     'reps': {'type': 'integer'

Что передавать модели:

- Описание инструмента `log_exercise_tool` в параметре `tools`.

- Системную инструкцию, которая объясняет роль ассистента.

- Сообщение пользователя, содержащее информацию об упражнении.

Модель проанализирует входные данные. Поскольку пользователь сообщил о выполненном упражнении, языковая модель хочет вызвать функцию `log_exercise` с параметрами, извлечёнными из сообщения:


In [11]:
print(f"Тип ответа: {res.output[0].type}")
print(f"Имя функции: {res.output[0].name}")
print(f"Аргументы: {res.output[0].arguments}")


Тип ответа: function_call
Имя функции: log_exercise
Аргументы: {"exercise_name": "приседания", "sets": 3, "reps": 12, "weight": 70}



### 4️⃣ Обработка ответа модели

Когда модель решает, что нужно вызвать функцию, она возвращает специальную структуру в ответе. Задача — найти этот вызов функции и выполнить его, а затем отправить результат обратно модели:


In [ ]:
# Проверяем, есть ли вызов функции в ответе
for output_item in res.output:
    if output_item.type == "function_call":
        # Извлекаем имя функции и аргументы
        function_name = output_item.name
        arguments_str = output_item.arguments  # Это строка в формате JSON
        
        print(f"Модель запросила вызов функции: {function_name}")
        print(f"С аргументами: {arguments_str}")
        
        # Парсим аргументы из JSON-строки в словарь Python
        arguments = json.loads(arguments_str)
        
        # Вызываем функцию с полученными аргументами
        if function_name == "log_exercise":
            result = log_exercise(**arguments)
            print(f"Функция выполнена, результат: {result}")
            
            # Формируем текстовое сообщение с результатом
            result_message = f"Результат выполнения функции {function_name}: {json.dumps(result, ensure_ascii=False)}"
            
            # Отправляем результат обратно модели
            follow_up = client.responses.create(
                model=model,
                store=True,
                previous_response_id=response.id,
                input=result_message
            )
            
            # Выводим финальный ответ модели пользователю
            print(follow_up.output_text)


Модель запросила вызов функции: log_exercise
С аргументами: {"exercise_name": "Приседания", "sets": 3, "reps": 12, "weight": 70}
Функция выполнена, результат: {'status': 'success', 'message': "Упражнение 'Приседания' успешно записано", 'record_id': 'c6ae7597-dbae-4220-a5d5-18f717ef9d82'}


Отлично! Твоё упражнение — **3 подхода по 12 приседаний с весом 70 кг** — успешно записано в дневник тренировок.  
ID записи: `c6ae7597-dbae-4220-a5d5-18f717ef9d82`

Продолжай в том же духе! Если хочешь, можешь добавить ещё одно упражнение или спланировать следующую тренировку. 💪


После выполнения этого кода модель получает результаты работы функции и формирует финальный ответ для пользователя с учётом успешности операции. Например, пользователь может получить сообщение: **Отлично! Я записал твои приседания: 3 подхода по 12 повторений с весом 70 кг. Продолжай в том же духе!**

❗ В примере передаётся `previous_response_id=response.id` при отправке результатов выполнения функции, чтобы модель сохраняла контекст диалога и могла сформировать последовательный ответ.

## Добавляем дополнительные функции

Теперь добавим боту несколько полезных инструментов: журнал тренировок, просмотр истории упражнений и расчёт сожжённых калорий. Это превратит простого текстового помощника в полноценный инструмент для отслеживания тренировок.

In [16]:
# Простое хранилище данных в памяти
exercises_db = {
    "exercise_log": []
}


# Функции для работы с данными - код повторяет то, что уже было выше
def log_exercise(exercise_name, sets, reps, weight=None, date=None):
    """
    Записывает информацию о выполненном упражнении в журнал тренировок
    
    Args:
        exercise_name (str): Название упражнения
        sets (int): Количество подходов
        reps (int): Количество повторений в каждом подходе
        weight (float, optional): Вес в кг
        date (str, optional): Дата тренировки в формате YYYY-MM-DD
    
    Returns:
        dict: Информация о записи с уникальным ID
    """
    # Генерируем уникальный ID для записи
    record_id = str(uuid.uuid4())
    
    # Устанавливаем текущую дату, если не указана
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')
    
    # Создаём запись
    record = {
        "id": record_id,
        "exercise": exercise_name,
        "sets": sets,
        "reps": reps,
        "weight": weight,
        "date": date
    }
    
    # В реальном приложении здесь был бы код для сохранения в базу данных
    # Для примера просто сохраняем в памяти
    if 'exercise_log' not in exercises_db:
        exercises_db['exercise_log'] = []
    
    exercises_db['exercise_log'].append(record)
    
    return {
        "status": "success",
        "message": f"Упражнение '{exercise_name}' успешно записано",
        "record_id": record_id
    }


def get_exercise_history(days=7):
    """
    Получает историю тренировок пользователя за указанное количество дней
    
    Args:
        user_id (str): Идентификатор пользователя
        days (int): Количество дней для истории
        
    Returns:
        list: Записи о тренировках
    """
    # Определите дату начала периода
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    print(start_date)
    
    # Отфильтруйте записи по пользователю и дате
    history = [
        record for record in exercises_db['exercise_log'] 
        if record.get('date') >= start_date
    ]
    
    return {
        "status": "success",
        "history": history
    }


def calculate_calories(exercise_name, duration_minutes, intensity="moderate", weight_kg=70):
    """
    Рассчитывает примерное количество сожжённых калорий
    
    Args:
        exercise_name (str): Название упражнения
        duration_minutes (int): Продолжительность в минутах
        intensity (str): Интенсивность (low, moderate, high)
        weight_kg (float): Вес пользователя в кг
        
    Returns:
        dict: Информация о сожжённых калориях
    """
    # Приблизительные значения MET (метаболический эквивалент задачи)
    # для различных упражнений и интенсивностей
    met_values = {
        "бег": {"low": 7, "moderate": 9, "high": 12},
        "ходьба": {"low": 3, "moderate": 4, "high": 5},
        "плавание": {"low": 5, "moderate": 7, "high": 10},
        "велосипед": {"low": 4, "moderate": 6, "high": 8},
        "приседания": {"low": 3, "moderate": 5, "high": 7},
        "отжимания": {"low": 3, "moderate": 5, "high": 8},
        # Для неизвестных упражнений
        "default": {"low": 3, "moderate": 5, "high": 7}
    }
    
    # Вы получите MET для указанного упражнения и интенсивности
    exercise_name_lower = exercise_name.lower()
    exercise_met = met_values.get(exercise_name_lower, met_values["default"])
    met = exercise_met.get(intensity, exercise_met["moderate"])
    
    # Формула для расчёта калорий: MET × вес (кг) × время (часы)
    calories = met * weight_kg * (duration_minutes / 60)
    
    return {
        "status": "success",
        "exercise": exercise_name,
        "duration_minutes": duration_minutes,
        "intensity": intensity,
        "calories_burned": round(calories, 1),
        "met_used": met
    }


Опишим эти функции для языковой модели, чтобы она могла их вызывать. В поле `description` опишем каждую функцию, а внутри структуры `parameters` — полностью все параметры:

In [15]:
tools = [
    {
        "type": "function",
        "name": "log_exercise",
        "description": "Записывает информацию о выполненном упражнении в журнал тренировок",
        "parameters": {
            "type": "object",
            "properties": {
                "exercise_name": {
                    "type": "string",
                    "description": "Название упражнения"
                },
                "sets": {
                    "type": "integer",
                    "description": "Количество подходов"
                },
                "reps": {
                    "type": "integer",
                    "description": "Количество повторений в каждом подходе"
                },
                "weight": {
                    "type": "number",
                    "description": "Вес в кг (если применимо)"
                },
                "date": {
                    "type": "string",
                    "description": "Дата тренировки в формате YYYY-MM-DD"
                }
            },
            "required": ["exercise_name", "sets", "reps"]
        }
    },
    {
        "type": "function",
        "name": "get_exercise_history",
        "description": "Получает историю тренировок пользователя за указанное количество дней",
        "parameters": {
            "type": "object",
            "properties": {
                "days": {
                    "type": "integer",
                    "description": "За сколько последних дней получить историю (по умолчанию 7)"
                }
            },
            "required": []
        }
    },
    {
        "type": "function",
        "name": "calculate_calories",
        "description": "Рассчитывает примерное количество сожжённых калорий во время тренировки",
        "parameters": {
            "type": "object",
            "properties": {
                "exercise_name": {
                    "type": "string",
                    "description": "Название упражнения"
                },
                "duration_minutes": {
                    "type": "integer",
                    "description": "Продолжительность упражнения в минутах"
                },
                "intensity": {
                    "type": "string",
                    "enum": ["low", "moderate", "high"],
                    "description": "Интенсивность тренировки: low (низкая), moderate (средняя), high (высокая)"
                },
                "weight_kg": {
                    "type": "number",
                    "description": "Вес пользователя в килограммах"
                }
            },
            "required": ["exercise_name", "duration_minutes"]
        }
    }
]

Для построения диалога обновим класс `Assistant` из предыдущих уроков — добавим поддержку инструментов и их обработку:


In [14]:
class Assistant:
    def __init__(self, instructions, model=model, tools=None, function_map=None):
        self.model = model
        self.instructions = instructions
        self.tools = tools or []
        self.previous_response_id_map = {}
        
        # Словарь с реализациями функций
        self.function_map = function_map
    
    def __call__(self, input_text, session_id='default'):
        """Обрабатывает сообщение пользователя и возвращает ответ"""
        previous_response_id = self.previous_response_id_map.get(session_id, None)
        
        # Вызываем модель с инструментами
        response = client.responses.create(
            model=self.model,
            store=True,
            previous_response_id=previous_response_id,
            instructions=self.instructions,
            input=input_text,
            tools=self.tools
        )
        
        # Обновляем ID ответа
        self.previous_response_id_map[session_id] = response.id
        
        # Обрабатываем ответ (включая возможные вызовы функций)
        return self._process_response(response, session_id)
    
    def _process_response(self, response, session_id):
        """Обрабатывает ответ модели, включая возможные вызовы функций"""
        
        # Проверяем наличие вызовов функций
        for output_item in response.output:
            if output_item.type == "function_call":
                # Извлекаем данные вызова
                function_name = output_item.name
                arguments_str = output_item.arguments
                
                # Парсим аргументы из JSON-строки
                function_args = json.loads(arguments_str)
                
                print(f"[DEBUG] Вызов функции: {function_name}({function_args})")
                
                # Вызываем функцию, если она есть в маппинге
                if function_name in self.function_map:
                    function_result = self.function_map[function_name](**function_args)
                    
                    print(f"[DEBUG] Результат функции: {function_result}")
                    
                    # Формируем сообщение с результатом
                    result_message = f"Результат выполнения функции {function_name}: {json.dumps(function_result, ensure_ascii=False)}"
                    
                    # Отправляем результат обратно модели
                    follow_up = client.responses.create(
                        model=self.model,
                        store=True,
                        previous_response_id=response.id,
                        input=result_message
                    )
                    
                    # Обновляем ID последнего ответа
                    self.previous_response_id_map[session_id] = follow_up.id
                    
                    # Рекурсивно обрабатываем новый ответ
                    # (модель может вызвать ещё одну функцию)
                    return self._process_response(follow_up, session_id)
        
        # Если вызовов функций нет, возвращаем текстовый ответ
        return response.output_text if hasattr(response, 'output_text') else ""


Создадим экземпляр ассистента с системным промтом, который описывает его роль и возможности:


In [17]:
instructions = """
Ты — профессиональный фитнес-ассистент спортивного клуба SuperGYM. 
Твоя задача — помогать пользователям:
1. Отвечать на вопросы о фитнесе, тренировках и здоровом образе жизни
2. Записывать информацию о выполненных упражнениях
3. Предоставлять историю тренировок
4. Рассчитывать сожжённые калории


Общайся энергично и мотивирующе. Предлагай конкретные рекомендации, 
основанные на данных пользователя.
"""

function_map={
            "log_exercise": log_exercise,
            "get_exercise_history": get_exercise_history,
            "calculate_calories": calculate_calories
}

fitness_assistant = Assistant(instructions, tools=tools, function_map=function_map)

Теперь мы можем взаимодействовать с нашим расширенным ассистентом:

In [18]:
print(fitness_assistant("Привет! Я сегодня сделал 4 подхода по 10 отжиманий. Запиши это."))

[DEBUG] Вызов функции: log_exercise({'exercise_name': 'отжимания', 'sets': 4, 'reps': 10, 'date': '2023-04-10'})
[DEBUG] Результат функции: {'status': 'success', 'message': "Упражнение 'отжимания' успешно записано", 'record_id': '850ccfe1-6b7d-47e5-ae06-1107cced725f'}
Отлично! Твои 4 подхода по 10 отжиманий успешно записаны ✅  
Продолжай в том же духе — ты молодец! 💪


In [19]:
print(fitness_assistant("Сколько калорий я сжёг за 30 минут бега с высокой интенсивностью?"))


[DEBUG] Вызов функции: calculate_calories({'exercise_name': 'бег', 'duration_minutes': 30, 'intensity': 'high', 'weight_kg': 70})
[DEBUG] Результат функции: {'status': 'success', 'exercise': 'бег', 'duration_minutes': 30, 'intensity': 'high', 'calories_burned': 420.0, 'met_used': 12}
За 30 минут бега с высокой интенсивностью (примерно 8–10 км/ч) ты сжёг около **420 калорий**.  

Это значение рассчитано приблизительно для человека весом 70 кг — если твой вес отличается, количество сожжённых калорий будет немного другим.  

Ты отлично потрудился — продолжай в том же темпе! 🏃‍♂️🔥


In [20]:
print(fitness_assistant("Мой вес 70 кг. Сколько калорий я тогда сжёг?"))


Отлично, что уточнил вес — и он как раз совпадает с тем, который использовался в расчёте! 🎯

Так что подтверждаем:  
При весе **70 кг**, за **30 минут бега с высокой интенсивностью**, ты сжёг **примерно 420 калорий**. ✅

Это мощная тренировка — гордись результатом! 💪🔥


In [21]:
print(exercises_db)

{'exercise_log': [{'id': '850ccfe1-6b7d-47e5-ae06-1107cced725f', 'exercise': 'отжимания', 'sets': 4, 'reps': 10, 'weight': None, 'date': '2023-04-10'}]}


In [22]:
print(fitness_assistant("Покажи историю моих тренировок"))
# Ассистент использует get_exercise_history и возвращает ответ

[DEBUG] Вызов функции: get_exercise_history({'days': 7})
2026-02-28
[DEBUG] Результат функции: {'status': 'success', 'history': []}
Пока в твоём журнале нет записей о предыдущих тренировках за последние 7 дней. 📅

Но не переживай — ты уже начал сильный старт: сегодня ты сделал **4 подхода по 10 отжиманий** и пробежал **30 минут с высокой интенсивностью**, сжигая около **420 калорий**! 🔥

Продолжай добавлять упражнения — и я буду вести полную историю твоих достижений. Ты на правильном пути! 💪


Благодаря Function Calling ассистент из простого собеседника превратился в полезный инструмент для отслеживания тренировок и расчёта физической активности. Он может не только давать советы, но и сохранять данные, анализировать их и предоставлять персонализированную информацию.

### Дополнительные задания

1️⃣ **Добавьте функцию для отслеживания веса и других параметров тела**. У пользователя должна быть возможность записать свой текущий вес, объёмы и другие метрики, а также просмотреть историю изменений.

2️⃣ **Реализуйте функцию для рекомендации упражнений**. На основе целей пользователя (похудение, набор массы, реабилитация) и доступного оборудования ассистент должен рекомендовать подходящие упражнения.

**3️⃣ Добавьте интеграцию с базой данных**. Вместо хранения данных в памяти используйте настоящую базу данных (SQL или NoSQL) для сохранения информации о пользователях и их тренировках.